In [2]:
import numpy as np
import pandas as pd
import os
import re
import statistics

from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import r2_score

from sklearn.neighbors import KNeighborsRegressor
import xgboost as xgb
from lightgbm import LGBMRegressor

In [3]:
yield_df = pd.read_csv("./data_1/all_yields_.csv")

In [4]:
yield_df.head()

,VRYIELDMAS,SECTIONID,Moisture,Elevation,xcoord,ycoord,area,Organic M,Ca,Mg,...,mean_cloudcover_month_7,mean_cloudcover_month_8,mean_cloudcover_month_9,mean_cloudcover_month_10,std_cloudcover_month_5,std_cloudcover_month_6,std_cloudcover_month_7,std_cloudcover_month_8,std_cloudcover_month_9,std_cloudcover_month_10
0,0.000000,499,14.38,269.856182,25.839963,48.834206,5.18,9.5,4901.0,100.0,...,67.474194,48.309677,59.55,47.425806,24.902733,24.864623,18.293386,24.280985,24.648883,34.1758
1,0.186290,499,14.41,269.818182,25.839974,48.834226,5.18,9.5,4901.0,100.0,...,67.474194,48.309677,59.55,47.425806,24.902733,24.864623,18.293386,24.280985,24.648883,34.1758
2,0.000000,499,14.44,269.751182,25.839979,48.834244,5.18,9.5,4901.0,100.0,...,67.474194,48.309677,59.55,47.425806,24.902733,24.864623,18.293386,24.280985,24.648883,34.1758
3,0.143233,499,14.45,269.748182,25.839986,48.834261,5.18,9.5,4901.0,100.0,...,67.474194,48.309677,59.55,47.425806,24.902733,24.864623,18.293386,24.280985,24.648883,34.1758
4,0.000000,499,14.24,269.713182,25.839996,48.834278,5.18,9.5,4901.0,100.0,...,67.474194,48.309677,59.55,47.425806,24.902733,24.864623,18.293386,24.280985,24.648883,34.1758


In [5]:
yield_df.columns

Index(['VRYIELDMAS', 'SECTIONID', 'Moisture', 'Elevation', 'xcoord', 'ycoord',
       'area', 'Organic M', 'Ca', 'Mg',
       ...
       'mean_cloudcover_month_7', 'mean_cloudcover_month_8',
       'mean_cloudcover_month_9', 'mean_cloudcover_month_10',
       'std_cloudcover_month_5', 'std_cloudcover_month_6',
       'std_cloudcover_month_7', 'std_cloudcover_month_8',
       'std_cloudcover_month_9', 'std_cloudcover_month_10'],
      dtype='object', length=172)

In [6]:
years = [2018, 2020, 2021, 2022, 2023, 2024]
yield_dfs = {}

for year in years:
    yield_dfs[year] = yield_df[yield_df['year'] == year]

In [7]:
for year in years:
    print(len(yield_dfs[year]))

26472
11715
38209
24897
43672
40167


In [8]:
feature_names = ["Moisture", "Elevation" , "xcoord" ,
                 "ycoord", "area", "Organic M", "Ca", "Mg", "Mn", "B",
                 "Cu", "Mo", "Fe", "Zn", "S", "P", "K", "Na", "pH", "C.E.C",
                 "year", "Crop_22", "Crop_34", "Crop_41", "Crop_45", "Crop_174", 

    "mean_tempmax_month_5", "mean_tempmax_month_6", "mean_tempmax_month_7",
    "mean_tempmax_month_8", "mean_tempmax_month_9", "mean_tempmax_month_10",
    "std_tempmax_month_5", "std_tempmax_month_6", "std_tempmax_month_7",
    "std_tempmax_month_8", "std_tempmax_month_9", "std_tempmax_month_10",

    "mean_tempmin_month_5", "mean_tempmin_month_6", "mean_tempmin_month_7",
    "mean_tempmin_month_8", "mean_tempmin_month_9", "mean_tempmin_month_10",
    "std_tempmin_month_5", "std_tempmin_month_6", "std_tempmin_month_7",
    "std_tempmin_month_8", "std_tempmin_month_9", "std_tempmin_month_10",

    "mean_temp_month_5", "mean_temp_month_6", "mean_temp_month_7",
    "mean_temp_month_8", "mean_temp_month_9", "mean_temp_month_10",
    "std_temp_month_5", "std_temp_month_6", "std_temp_month_7",
    "std_temp_month_8", "std_temp_month_9", "std_temp_month_10",
    
    "mean_humidity_month_5", "mean_humidity_month_6", "mean_humidity_month_7",
    "mean_humidity_month_8", "mean_humidity_month_9", "mean_humidity_month_10",
    "std_humidity_month_5", "std_humidity_month_6", "std_humidity_month_7",
    "std_humidity_month_8", "std_humidity_month_9", "std_humidity_month_10",
    
    "mean_precip_month_5", "mean_precip_month_6", "mean_precip_month_7",
    "mean_precip_month_8", "mean_precip_month_9", "mean_precip_month_10",
    "std_precip_month_5", "std_precip_month_6", "std_precip_month_7",
    "std_precip_month_8", "std_precip_month_9", "std_precip_month_10",
    
    "mean_cloudcover_month_5", "mean_cloudcover_month_6", "mean_cloudcover_month_7",
    "mean_cloudcover_month_8", "mean_cloudcover_month_9", "mean_cloudcover_month_10",
    "std_cloudcover_month_5", "std_cloudcover_month_6", "std_cloudcover_month_7",
    "std_cloudcover_month_8", "std_cloudcover_month_9", "std_cloudcover_month_10",
    ]

In [13]:
X_trains = []
X_tests = []
y_trains = []
y_tests = []

for year in years:
    X = yield_dfs[year][feature_names]
    y = yield_dfs[year]["VRYIELDMAS"]

    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
    X_trains.append(X_train)
    X_tests.append(X_test)
    y_trains.append(y_train)
    y_tests.append(y_test)

### Light GBM

In [ ]:
models_gbm = []

for i in range(6):
    model_gbm = LGBMRegressor(learning_rate=0.1, max_depth=10, n_estimators=100, num_leaves=64)
    model_gbm.fit(X_trains[i], y_trains[i])
    models_gbm.append(model_gbm)

In [16]:
mae_gbm = []
mse_gbm = []
rmse_gbm = []
r_2_gbm = []

for i in range(6):
    y_pred = models_gbm[i].predict(X_tests[i])

    mae = metrics.mean_absolute_error(y_tests[i], y_pred)
    mae_gbm.append(mae)
    mse = metrics.mean_squared_error(y_tests[i], y_pred)
    mse_gbm.append(mse)
    rmse = np.sqrt(metrics.mean_squared_error(y_tests[i], y_pred))
    rmse_gbm.append(rmse)
    r_2 = r2_score(y_tests[i], y_pred)
    r_2_gbm.append(r_2)

    print("Year: ", years[i])
    print('Mean Absolute Error:', mae)
    print('Mean Square Error:', mse)
    print('Root Mean Square Error:', rmse)
    print('R^2:', r_2)
    print("                 ")

print("Mean MAE: ", statistics.mean(mae_gbm))
print("STD MAE: ", statistics.stdev(mae_gbm))

print("Mean MSE: ", statistics.mean(mse_gbm))
print("STD MSE: ", statistics.stdev(mse_gbm))

print("Mean RMSE: ", statistics.mean(rmse_gbm))
print("STD RMSE: ", statistics.stdev(rmse_gbm))

print("Mean r2: ", statistics.mean(r_2_gbm))
print("STD r2: ", statistics.stdev(r_2_gbm))

Year:  2018
Mean Absolute Error: 0.24340623954091695
Mean Square Error: 0.10493945069522206
Root Mean Square Error: 0.32394359184157673
R^2: 0.7959866132545221
                 
Year:  2020
Mean Absolute Error: 0.4828610518351193
Mean Square Error: 0.5343754617801457
Root Mean Square Error: 0.7310098917115594
R^2: 0.7573446125426412
                 
Year:  2021
Mean Absolute Error: 0.9392536906892832
Mean Square Error: 1.8109261372908076
Root Mean Square Error: 1.3457065569026583
R^2: 0.65499737590065
                 
Year:  2022
Mean Absolute Error: 0.3451387729800022
Mean Square Error: 0.2870614800261025
Root Mean Square Error: 0.5357811867041455
R^2: 0.7384778827568983
                 
Year:  2023
Mean Absolute Error: 0.7118261239784086
Mean Square Error: 0.9977911064892052
Root Mean Square Error: 0.9988949426687499
R^2: 0.6565915113782228
                 
Year:  2024
Mean Absolute Error: 0.30956988772543875
Mean Square Error: 0.18761932683154692
Root Mean Square Error: 0.433150

c:\Users\katja\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\katja\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\katja\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\katja\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\katja\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\d

### XGBoost

In [30]:
models_xgb = []

for i in range(6):
    regressor = xgb.XGBRegressor(learning_rate = 0.01,
                           n_estimators  = 700,
                           max_depth     = 10,
                           eval_metric   = 'rmse')
    regressor.fit(X_trains[i], y_trains[i])
    models_xgb.append(regressor)

In [38]:
mae_xgb = []
mse_xgb = []
rmse_xgb = []
r_2_xgb = []

for i in range(6):
    y_pred = models_xgb[i].predict(X_tests[i])

    mae = metrics.mean_absolute_error(y_tests[i], y_pred)
    mae_xgb.append(mae)
    mse = metrics.mean_squared_error(y_tests[i], y_pred)
    mse_xgb.append(mse)
    rmse = np.sqrt(metrics.mean_squared_error(y_tests[i], y_pred))
    rmse_xgb.append(rmse)
    r_2 = r2_score(y_tests[i], y_pred)
    r_2_xgb.append(r_2)

    print("Year: ", years[i])
    print('Mean Absolute Error:', mae)
    print('Mean Square Error:', mse)
    print('Root Mean Square Error:', rmse)
    print('R^2:', r_2)
    print("                 ")

print("Mean MAE: ", statistics.mean(mae_xgb))
print("STD MAE: ", statistics.stdev(mae_xgb))

print("Mean MSE: ", statistics.mean(mse_xgb))
print("STD MSE: ", statistics.stdev(mse_xgb))

print("Mean RMSE: ", statistics.mean(rmse_xgb))
print("STD RMSE: ", statistics.stdev(rmse_xgb))

print("Mean r2: ", statistics.mean(r_2_xgb))
print("STD r2: ", statistics.stdev(r_2_xgb))

Year:  2018
Mean Absolute Error: 0.21883836286052227
Mean Square Error: 0.08782909368832112
Root Mean Square Error: 0.2963597369554797
R^2: 0.8292509562473239
                 
Year:  2020
Mean Absolute Error: 0.3968331362675376
Mean Square Error: 0.41480432299280784
Root Mean Square Error: 0.6440530436173777
R^2: 0.8116408575732492
                 
Year:  2021
Mean Absolute Error: 0.7897989416235096
Mean Square Error: 1.382854570155331
Root Mean Square Error: 1.1759483705313474
R^2: 0.7365500195578907
                 
Year:  2022
Mean Absolute Error: 0.29105365414343826
Mean Square Error: 0.23062504875121329
Root Mean Square Error: 0.4802343685651968
R^2: 0.7898932624703721
                 
Year:  2023
Mean Absolute Error: 0.6358178450851217
Mean Square Error: 0.8266513857675931
Root Mean Square Error: 0.9092037097194408
R^2: 0.7154924501157418
                 
Year:  2024
Mean Absolute Error: 0.2592684781572964
Mean Square Error: 0.14216977462063554
Root Mean Square Error: 0.3770

### KNN

In [32]:
models_knn = []

for i in range(6):
    knn_reg = KNeighborsRegressor(n_neighbors=3)
    knn_reg.fit(X_trains[i], y_trains[i])
    models_knn.append(knn_reg)

In [39]:
mae_knn = []
mse_knn = []
rmse_knn = []
r_2_knn = []

for i in range(6):
    os.environ['OMP_NUM_THREADS'] = '1'
    y_pred = models_knn[i].predict(X_tests[i])

    mae = metrics.mean_absolute_error(y_tests[i], y_pred)
    mae_knn.append(mae)
    mse = metrics.mean_squared_error(y_tests[i], y_pred)
    mse_knn.append(mse)
    rmse = np.sqrt(metrics.mean_squared_error(y_tests[i], y_pred))
    rmse_knn.append(rmse)
    r_2 = r2_score(y_tests[i], y_pred)
    r_2_knn.append(r_2)

    print("Year: ", years[i])
    print('Mean Absolute Error:', mae)
    print('Mean Square Error:', mse)
    print('Root Mean Square Error:', rmse)
    print('R^2:', r_2)
    print("                 ")

print("Mean MAE: ", statistics.mean(mae_knn))
print("STD MAE: ", statistics.stdev(mae_knn))

print("Mean MSE: ", statistics.mean(mse_knn))
print("STD MSE: ", statistics.stdev(mse_knn))

print("Mean RMSE: ", statistics.mean(rmse_knn))
print("STD RMSE: ", statistics.stdev(rmse_knn))

print("Mean r2: ", statistics.mean(r_2_knn))
print("STD r2: ", statistics.stdev(r_2_knn))

Year:  2018
Mean Absolute Error: 0.18672071510481586
Mean Square Error: 0.07099349839432552
Root Mean Square Error: 0.266446051564525
R^2: 0.8619811334213946
                 
Year:  2020
Mean Absolute Error: 0.32113248062028743
Mean Square Error: 0.3799772785970484
Root Mean Square Error: 0.6164229705300155
R^2: 0.8274555245186498
                 
Year:  2021
Mean Absolute Error: 0.8382312045293554
Mean Square Error: 1.8004690077238152
Root Mean Square Error: 1.3418155639743545
R^2: 0.6569895814726324
                 
Year:  2022
Mean Absolute Error: 0.2251793673641232
Mean Square Error: 0.18300770551982118
Root Mean Square Error: 0.42779399892918224
R^2: 0.833274173131854
                 
Year:  2023
Mean Absolute Error: 0.6456524695523755
Mean Square Error: 0.9379163137032936
Root Mean Square Error: 0.9684607961622884
R^2: 0.6771985422120604
                 
Year:  2024
Mean Absolute Error: 0.14878555771014856
Mean Square Error: 0.0733420467828523
Root Mean Square Error: 0.27081